In [1]:
import sounddevice as sd

print(sd.query_devices())

   0 bcm2835 HDMI 1: - (hw:0,0), ALSA (0 in, 8 out)
   1 bcm2835 Headphones: - (hw:1,0), ALSA (0 in, 8 out)
   2 USB 2.0 Camera: Audio (hw:2,0), ALSA (1 in, 0 out)
   3 USB PnP Audio Device: Audio (hw:3,0), ALSA (2 in, 2 out)
   4 sysdefault, ALSA (0 in, 128 out)
   5 samplerate, ALSA (0 in, 128 out)
   6 speexrate, ALSA (0 in, 128 out)
   7 pulse, ALSA (32 in, 32 out)
   8 upmix, ALSA (0 in, 8 out)
   9 vdownmix, ALSA (0 in, 6 out)
  10 dmix, ALSA (0 in, 2 out)
* 11 default, ALSA (32 in, 32 out)
  12 /dev/dsp, OSS (0 in, 16 out)
  13 /dev/dsp1, OSS (0 in, 16 out)
  14 /dev/dsp2, OSS (16 in, 0 out)
  15 /dev/dsp3, OSS (16 in, 16 out)


In [2]:
import sounddevice as sd
import queue
import json
import time

from vosk import Model, KaldiRecognizer
import Arm_Lib

In [3]:
Arm = Arm_Lib.Arm_Device()

HOME = [90,135,0,0,90,10]

Arm.Arm_serial_servo_write6_array(HOME,1000)

In [4]:
def home():
    print("Home")
    Arm.Arm_serial_servo_write6_array(HOME,1000)


def open_gripper():
    print("Open Gripper")
    Arm.Arm_serial_servo_write(6,60,500)


def close_gripper():
    print("Close Gripper")
    Arm.Arm_serial_servo_write(6,180,500)

In [5]:
def front_back():

    print("front back")

    for i in range(2):

        Arm.Arm_serial_servo_write(2,50,400)
        time.sleep(0.5)

        Arm.Arm_serial_servo_write(2,130,400)
        time.sleep(0.5)

    home()

In [6]:
model = Model("vosk-model-small-en-us-0.15")

q = queue.Queue()

def callback(indata, frames, time, status):
    q.put(bytes(indata))

mic_index =3
device_info = sd.query_devices(mic_index, 'input')
samplerate = int(device_info['default_samplerate'])

rec = KaldiRecognizer(
    model,
    samplerate,
    '["front_back"]'
)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-small-en-us-0.15/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from vosk-model-small-en-us-0.15/graph/HCLr.fst vosk-model-small-en-us-0.15/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-small-en-us-0.15/graph/phones/word_boundary.int
LOG (VoskAPI:UpdateGrammarFst():recognizer.cc:287) ["front_back"]
WARNING (VoskAPI:UpdateGrammarFst(

In [7]:
running = True

last_command = ""

def execute_command(text):

    global running
    global last_command

    text = text.lower().strip()

    if text == last_command:
        return

    last_command = text

    print("Command :", text)

    if "front_back" in text:
        front_back()

    else:
        print("Unknown command")

In [ ]:
with sd.RawInputStream(
    device=mic_index,
    samplerate=samplerate,
    blocksize=4096,
    dtype='int16',
    channels=1,
    callback=callback):
    print("----------------------------------")
    print("Voice Control Started")
    print("----------------------------------")
    print("Commands:")
    print("front_back")
    print("----------------------------------")

    while running:
        data = q.get()

        if rec.AcceptWaveform(data):
            result = json.loads(rec.Result())
            text = result.get("text", "")

            if text:
                print("🧠 Heard:", text)
                execute_command(text)

print("Listener stopped")
home()

del Arm

----------------------------------
Voice Control Started
----------------------------------
Commands:
front_back
----------------------------------
